# HPO после model selection

Model selection уже выбрал две комбинации: `XGBRegressor` и frozen `LightGBM` с `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES`. HPO проверяет, можно ли улучшить **каждую из этих моделей** на той же temporal validation. Другие модели и feature configurations не тюним. Accepted feature registry не меняется.

Notebook действительно запускает Optuna trials и три обучения на каждом trial. Он использует общий код `ml/src/hyperparameter_tuning.py` и `ml/src/model_selection.py`, поэтому CLI и notebook выполняют одинаковый эксперимент. Здесь выбираем конфигурацию для будущего Jan–Oct → Nov–Dec fit; сам финальный fit не выполняем.

## 1. Исторические данные и неизменный контракт

Загружаем текущие model-ready historical CSV с weather columns и сверяем их SHA-256 с результатом model selection. Единица прогноза — `(route, date, hour)`, target — `boardings`. Признаки фиксированы: accepted core, пять прежних calendar candidates и `temperature_bucket`. Предобработка categorical features остаётся прежней: pandas categories для LightGBM и XGBoost. Новых features и target-derived лагов нет.

Nov–Dec данные не читаются: фактического target за этот период нет, поэтому они не могут участвовать в objective или выборе trial.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/src/hyperparameter_tuning.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Project root not found')
sys.path.insert(0, str(root / 'ml/src'))
import hyperparameter_tuning as hpo
import model_selection as ms

rows, folds, reference = hpo.load_inputs()
display(pd.DataFrame([{'historical_rows': len(rows), 'target': 'boardings',
                       'feature_count': len(ms.FEATURE_CONFIGURATIONS[hpo.CONFIGURATION]),
                       'selected_models': ', '.join(hpo.FAMILIES),
                       'data_hashes_match': True}]))

C:\Users\sirac\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,historical_rows,target,feature_count,selected_models,data_hashes_match
0,72960,boardings,12,"XGBRegressor, LightGBM",True


## 2. Temporal folds и pooled objective

Каждый trial обучается на Jan–Apr и прогнозирует May–Jun; затем на Jan–Jun → Jul–Aug; затем на Jan–Aug → Sep–Oct. Это три независимых временных holdout-прогноза. Random CV смешал бы прошлое и будущее.

`WAPE = Σ|actual − prediction| / Σactual`. Objective Optuna — **pooled WAPE**: сумма абсолютных ошибок трёх непересекающихся holdouts, делённая на сумму их фактических посадок. Оптимизация только по Sep–Oct сделала бы параметры зависимыми от одного сезона. Fold WAPE, mean/median, pooled absolute error, per-route WAPE и время сохраняются для каждого trial. Route 5 после `predict` всегда получает `0`, как в model selection.

In [2]:
display(pd.DataFrame([{'fold': spec[0],
                       'train': f'{spec[1]} … {spec[2]}',
                       'validation': f'{spec[3]} … {spec[4]}',
                       'train_rows': len(train), 'validation_rows': len(valid)}
                      for spec, (train, valid) in zip(ms.FOLDS, folds)]))
display(pd.DataFrame([{'family': family,
                       'frozen pooled WAPE': reference['summaries'][family][hpo.CONFIGURATION]['pooled_wape']}
                      for family in hpo.FAMILIES]).round(6))

,fold,train,validation,train_rows,validation_rows
0,jan_apr_to_may_jun,2025-01-01 … 2025-04-30,2025-05-01 … 2025-06-30,28800,14640
1,jan_jun_to_jul_aug,2025-01-01 … 2025-06-30,2025-07-01 … 2025-08-31,43440,14880
2,jan_aug_to_sep_oct,2025-01-01 … 2025-08-31,2025-09-01 … 2025-10-31,58320,14640


,family,frozen pooled WAPE
0,XGBRegressor,0.153647
1,LightGBM,0.155712


## 3. Ограниченный search space

Optuna `TPESampler(seed=42)` проводит 16 trials на модель: trial 0 точно повторяет frozen baseline, ещё 15 исследуют заданный дискретный search space. Первые пять trials служат стартовой выборкой TPE. Это ограниченный search, а не полный grid.

Для обеих моделей `n_estimators` задаёт число деревьев, `max_depth` ограничивает сложность дерева, `learning_rate` регулирует шаг обучения. У XGBoost `min_child_weight` ограничивает создание слабоподдержанных листьев. У LightGBM `num_leaves` ограничивает число листьев, `min_child_samples` — минимальное число строк в листе. Диапазоны находятся вокруг frozen 300 деревьев, depth 6 и learning rate 0.1; они подходят для десятков тысяч строк и небольшого фиксированного набора признаков. Objective модели, random seed, categorical treatment и postprocessing остаются прежними.

In [3]:
display(pd.DataFrame([{'family': family, 'parameter': name,
                       'values': ', '.join(map(str, values)),
                       'frozen': hpo.FROZEN_SEARCH_PARAMS[family][name]}
                      for family, space in hpo.SEARCH_SPACES.items()
                      for name, values in space.items()]))
print('Trials per model:', hpo.TRIALS_PER_FAMILY,
      '| sampler seed:', hpo.SEED,
      '| feature configuration:', hpo.CONFIGURATION)

,family,parameter,values,frozen
0,XGBRegressor,n_estimators,"200, 300, 400, 500",300.0
1,XGBRegressor,max_depth,"4, 5, 6, 7",6.0
2,XGBRegressor,learning_rate,"0.05, 0.075, 0.1, 0.125",0.1
3,XGBRegressor,min_child_weight,"1, 3, 6, 10",1.0
4,LightGBM,n_estimators,"200, 300, 400, 500",300.0
5,LightGBM,max_depth,"6, 7, 8",6.0
6,LightGBM,learning_rate,"0.05, 0.075, 0.1, 0.125",0.1
7,LightGBM,num_leaves,"15, 31, 47, 63",31.0
8,LightGBM,min_child_samples,"10, 20, 40",20.0


Trials per model: 16 | sampler seed: 42 | feature configuration: calendar_weather


## 4. XGBoost trials

Сначала запускаем frozen control trial и сверяем абсолютные ошибки каждого fold и маршрута с model selection. При расхождении search останавливается. Затем Optuna проверяет 15 комбинаций. Таблица показывает лучшие trials по pooled WAPE, а не отдельный Sep–Oct leaderboard. Время `fit` и `predict` суммируется по трём folds.

In [4]:
xgb_study = hpo.run_study(folds, reference, 'XGBRegressor')

def compact_trials(study, count=5):
    ranked = sorted(study['trials'], key=lambda trial: trial['summary']['pooled_wape'])[:count]
    return pd.DataFrame([{'trial': trial['number'], 'kind': trial['kind'],
                          'pooled WAPE': trial['summary']['pooled_wape'],
                          'May–Jun': trial['fold_results'][0]['wape'],
                          'Jul–Aug': trial['fold_results'][1]['wape'],
                          'Sep–Oct': trial['fold_results'][2]['wape'],
                          'fit s': trial['summary']['train_seconds'],
                          'predict s': trial['summary']['inference_seconds']}
                         for trial in ranked]).round(6)

display(compact_trials(xgb_study))
print('Frozen trial reproduced:', xgb_study['frozen']['number'] == 0,
      '| best sampled trial:', xgb_study['best_tuned']['number'])

,trial,kind,pooled WAPE,May–Jun,Jul–Aug,Sep–Oct,fit s,predict s
0,11,sampled,0.153481,0.153758,0.198392,0.115491,33.004460,0.999371
1,14,sampled,0.153481,0.153758,0.198392,0.115491,34.471929,1.038998
2,0,frozen_control,0.153647,0.153780,0.198772,0.115607,19.915353,0.636597
3,5,sampled,0.153647,0.153780,0.198772,0.115607,18.367352,0.623517
4,15,sampled,0.153686,0.152355,0.200513,0.115543,31.366169,1.041747


Frozen trial reproduced: True | best sampled trial: 11


## 5. LightGBM trials

Применяем ту же pooled objective и те же folds ко второму HPO candidate. Frozen LightGBM также должен пройти контроль воспроизведения до новых trials. Сравниваем семейства только после завершения обоих searches.

In [5]:
lgb_study = hpo.run_study(folds, reference, 'LightGBM')
display(compact_trials(lgb_study))
print('Frozen trial reproduced:', lgb_study['frozen']['number'] == 0,
      '| best sampled trial:', lgb_study['best_tuned']['number'])

,trial,kind,pooled WAPE,May–Jun,Jul–Aug,Sep–Oct,fit s,predict s
0,8,sampled,0.150956,0.141207,0.205422,0.114021,3.166299,0.696030
1,13,sampled,0.151600,0.141113,0.206385,0.115065,1.469554,0.299470
2,10,sampled,0.152079,0.142500,0.205851,0.115574,2.908924,0.683668
3,11,sampled,0.152079,0.142500,0.205851,0.115574,2.931336,0.704334
4,15,sampled,0.152786,0.145374,0.205716,0.115023,1.519075,0.317301


Frozen trial reproduced: True | best sampled trial: 8


## 6. Frozen против best tuned

`best tuned` — sampled trial с минимальным pooled WAPE для каждой модели. HPO полезен только при отрицательном `Δ WAPE = tuned − frozen`. Если улучшения нет, дальше передаётся frozen конфигурация. Смотрим также fold и per-route WAPE: небольшое pooled улучшение может скрывать нестабильность. Route 5 в маршрутной таблице имеет неопределённый WAPE при нулевом знаменателе; сами прогнозы равны `0`.

In [6]:
studies = {'XGBRegressor': xgb_study, 'LightGBM': lgb_study}
result = hpo.build_result(studies)
comparison, routes = [], []
for family, study in studies.items():
    baseline = study['frozen']['summary']['pooled_wape']
    for label, trial in (('frozen', study['frozen']), ('best tuned', study['best_tuned'])):
        summary = trial['summary']
        comparison.append({'family': family, 'configuration': label, 'trial': trial['number'],
                           'May–Jun WAPE': trial['fold_results'][0]['wape'],
                           'Jul–Aug WAPE': trial['fold_results'][1]['wape'],
                           'Sep–Oct WAPE': trial['fold_results'][2]['wape'],
                           'pooled WAPE': summary['pooled_wape'],
                           'Δ vs frozen': summary['pooled_wape'] - baseline,
                           'pooled absolute error': summary['pooled_absolute_error'],
                           'mean WAPE': summary['mean_wape'],
                           'median WAPE': summary['median_wape'],
                           'fit s': summary['train_seconds'],
                           'predict s': summary['inference_seconds']})
        for route, values in summary['per_route'].items():
            routes.append({'family': family, 'configuration': label,
                           'route': route, 'WAPE': values['wape']})
display(pd.DataFrame(comparison).round(6))
display(pd.DataFrame(routes).pivot(index=['family', 'configuration'],
                                   columns='route', values='WAPE').round(4))
display(pd.DataFrame([{'family': family, 'best tuned trial': study['best_tuned']['number'],
                       'best tuned params': study['best_tuned']['selected_parameters'],
                       'HPO useful': study['hpo_useful']}
                      for family, study in studies.items()]))

,family,configuration,trial,May–Jun WAPE,Jul–Aug WAPE,Sep–Oct WAPE,pooled WAPE,Δ vs frozen,pooled absolute error,mean WAPE,median WAPE,fit s,predict s
0,XGBRegressor,frozen,0,0.153780,0.198772,0.115607,0.153647,0.000000,5381944,0.156053,0.153780,19.915353,0.636597
1,XGBRegressor,best tuned,11,0.153758,0.198392,0.115491,0.153481,-0.000166,5376138,0.155880,0.153758,33.004460,0.999371
2,LightGBM,frozen,0,0.156927,0.201950,0.115756,0.155712,0.000000,5454275,0.158211,0.156927,2.691065,0.676775
3,LightGBM,best tuned,8,0.141207,0.205422,0.114021,0.150956,-0.004756,5287697,0.153550,0.141207,3.166299,0.696030


route                            1      11      12      17      25      26  \
family       configuration                                                   
LightGBM     best tuned     0.1249  0.1261  0.1215  0.1019  0.1935  0.1657   
             frozen         0.1261  0.1284  0.1204  0.1200  0.1956  0.1651   
XGBRegressor best tuned     0.1222  0.1255  0.1184  0.1135  0.1960  0.1681   
             frozen         0.1223  0.1258  0.1185  0.1139  0.1963  0.1686   

route                           28   5      50       7  
family       configuration                              
LightGBM     best tuned     0.1844 NaN  0.2123  0.2714  
             frozen         0.1840 NaN  0.2148  0.2696  
XGBRegressor best tuned     0.1844 NaN  0.2158  0.2702  
             frozen         0.1849 NaN  0.2157  0.2698

,family,best tuned trial,best tuned params,HPO useful
0,XGBRegressor,11,"{'n_estimators': 500, 'max_depth': 6, 'learnin...",True
1,LightGBM,8,"{'n_estimators': 400, 'max_depth': 8, 'learnin...",True


## 7. Решение и provenance

Для будущего Jan–Oct → Nov–Dec fit сравниваем best tuned каждой модели, если он улучшил её frozen pooled WAPE; иначе frozen. Меньший pooled WAPE выбирает основную модель. Вторая остаётся `ensemble candidate`, если её результат отличается не более чем на заранее заданные 0.005 абсолютного WAPE. Это пометка для отдельного эксперимента, не запуск ensemble.

Сохраняем все trials, параметры, метрики, время, версии библиотек и SHA-256 кода/данных в experiment infrastructure. Nov–Dec не участвовал в objective; финального fit здесь нет.

In [7]:
hpo.save_result(result)
print('Saved:', hpo.ARTIFACT.relative_to(root).as_posix())
print('Final configuration:', result['final_configuration'])
print('Ensemble candidate:', result['ensemble_candidate'])

Saved: ml/experiments/results/hyperparameter_tuning.json
Final configuration: {'family': 'LightGBM', 'kind': 'sampled', 'parameters': {'objective': 'regression_l1', 'n_estimators': 400, 'max_depth': 8, 'num_leaves': 15, 'learning_rate': 0.1, 'random_state': 42, 'n_jobs': 4, 'verbosity': -1, 'min_child_samples': 40}, 'pooled_wape': 0.1509561111066385}
Ensemble candidate: {'family': 'XGBRegressor', 'kind': 'sampled', 'parameters': {'objective': 'reg:absoluteerror', 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.1, 'tree_method': 'hist', 'enable_categorical': True, 'random_state': 42, 'n_jobs': 4, 'min_child_weight': 1}, 'pooled_wape': 0.1534809739008535}


## Вывод HPO stage

Оба sampled best улучшили соответствующий frozen pooled WAPE. Для Jan–Oct → Nov–Dec final fit выбрана **LightGBM, trial 8**: `n_estimators=400`, `max_depth=8`, `learning_rate=0.1`, `num_leaves=15`, `min_child_samples=40`; остальные параметры и features прежние. Pooled WAPE снизился с `0.155712` до `0.150956`. При этом Jul–Aug WAPE вырос с `0.201950` до `0.205422`, поэтому результат не означает улучшения каждого сезона.

**XGBRegressor, trial 11** (`n_estimators=500`, `max_depth=6`, `learning_rate=0.1`, `min_child_weight=1`) снизил pooled WAPE с `0.153647` до `0.153481` и остаётся `ensemble candidate`: разрыв с выбранной LightGBM равен примерно `0.002525`, меньше заранее заданного порога `0.005`. Выигрыш XGBoost очень мал, поэтому его не следует считать доказательством устойчивого улучшения за пределами этих folds. Выигрыш на одном Sep–Oct без pooled улучшения не считался бы полезным HPO. Детали trials зафиксированы в `ml/experiments/results/hyperparameter_tuning.json` и `.codex-local/DECISIONS.md`.